# 02 — Survival Analysis
How long do users survive in the funnel?
Uses `lifelines` for Kaplan-Meier curves and Cox Proportional Hazards.

In [1]:
import duckdb
import pandas as pd
import numpy as np
import os
from pathlib import Path

def find_project_root() -> Path:
    start = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
    for path in (start, *start.parents):
        if (path / "requirements.txt").exists() and (path / "data").exists():
            return path
    raise RuntimeError("Could not locate project root.")

ROOT = find_project_root()
MPLCONFIGDIR = ROOT / ".matplotlib"
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIGDIR))

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from lifelines import KaplanMeierFitter, CoxPHFitter

sns.set_theme(style="whitegrid")

DATA_PATH = ROOT / "data" / "funnel_events.csv"
ASSETS_DIR = ROOT / "app" / "assets"
ASSETS_DIR.mkdir(parents=True, exist_ok=True)

STEP_ORDER = ["zip", "vehicle", "driver", "quotes", "bind"]
STEP_TO_INT = {s: i + 1 for i, s in enumerate(STEP_ORDER)}

# ── Load & prep ───────────────────────────────────────────────────────────────

In [2]:
con = duckdb.connect()
df = con.execute(f"SELECT * FROM read_csv_auto('{DATA_PATH}')").df()

# Build one row per session: duration = deepest step reached, event = dropped
session = (
    df.sort_values("step_order")
    .groupby("session_id")
    .agg(
        duration=("step_order", "max"),
        event=("dropped", "any"),           # True if user dropped anywhere
        channel=("channel", "first"),
        device=("device", "first"),
        prior_insured=("prior_insured", "first"),
    )
    .reset_index()
)
session["event"] = session["event"].astype(int)
print(session.shape)
session.head(3)

(50000, 6)


,session_id,duration,event,channel,device,prior_insured
0,0,3,1,paid_search,tablet,False
1,1,2,1,paid_search,desktop,True
2,2,5,0,organic,mobile,True


## 1. Overall Kaplan-Meier Survival Curve

In [3]:
kmf = KaplanMeierFitter()
kmf.fit(durations=session["duration"], event_observed=session["event"])

fig, ax = plt.subplots(figsize=(8, 5))
kmf.plot_survival_function(ax=ax, ci_show=True, label="All users")
ax.set_xticks(range(1, 6))
ax.set_xticklabels(STEP_ORDER)
ax.set_xlabel("Funnel Step")
ax.set_ylabel("Probability of Still Being in Funnel")
ax.set_title("Kaplan-Meier: User Survival Through Quote Funnel", fontsize=13)
plt.tight_layout()
plt.savefig(ASSETS_DIR / "km_overall.png", dpi=150)
plt.close()

## 2. KM Curves by Channel

In [4]:
fig, ax = plt.subplots(figsize=(9, 5))
for channel, grp in session.groupby("channel"):
    kmf_c = KaplanMeierFitter()
    kmf_c.fit(grp["duration"], grp["event"], label=channel)
    kmf_c.plot_survival_function(ax=ax, ci_show=False)

ax.set_xticks(range(1, 6))
ax.set_xticklabels(STEP_ORDER)
ax.set_xlabel("Funnel Step")
ax.set_ylabel("Survival Probability")
ax.set_title("KM Curves by Acquisition Channel", fontsize=13)
plt.tight_layout()
plt.savefig(ASSETS_DIR / "km_by_channel.png", dpi=150)
plt.close()

## 3. KM Curves by Device

In [5]:
fig, ax = plt.subplots(figsize=(9, 5))
for device, grp in session.groupby("device"):
    kmf_d = KaplanMeierFitter()
    kmf_d.fit(grp["duration"], grp["event"], label=device)
    kmf_d.plot_survival_function(ax=ax, ci_show=False)

ax.set_xticks(range(1, 6))
ax.set_xticklabels(STEP_ORDER)
ax.set_xlabel("Funnel Step")
ax.set_ylabel("Survival Probability")
ax.set_title("KM Curves by Device Type", fontsize=13)
plt.tight_layout()
plt.savefig(ASSETS_DIR / "km_by_device.png", dpi=150)
plt.close()

## 4. Cox Proportional Hazards Model
Quantify the effect of each feature on the hazard (risk of abandoning).

In [6]:
cox_df = session.copy()

# Encode categoricals as dummies (drop first to avoid multicollinearity)
cox_df = pd.get_dummies(cox_df, columns=["channel", "device"], drop_first=True)
cox_df = cox_df.drop(columns=["session_id"])

cph = CoxPHFitter()
cph.fit(cox_df, duration_col="duration", event_col="event")
cph.print_summary()

<lifelines.CoxPHFitter: fitted with 50000 total observations, 16529 right-censored observations>
             duration col = 'duration'
                event col = 'event'
      baseline estimation = breslow
   number of observations = 50000
number of events observed = 33471
   partial log-likelihood = -345210.81
         time fit was run = 2026-06-25 09:26:00 UTC

---
                     coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                            
prior_insured       -0.00      1.00      0.01           -0.03            0.02                0.97                1.02
channel_organic     -0.20      0.82      0.02           -0.24           -0.17                0.79                0.85
channel_paid_search  0.27      1.31      0.02            0.23            0.30                1.26                1.35
channel_referral    -0.12      0.89      0.02           -0.16           -0.08                0.85                0.92
device_mobile        0.55      1.74      0.01            0.53            0.58                1.70                1.78
device_tablet        0.22      1.25      0.02            0.18            0.27                1.19                1.31

                     cmp to      z      p  -log2(p)
covariate                                          
prior_insured          0.00  -0.37   0.71      0.50
channel_organic        0.00 -11.81 <0.005    104.48
channel_paid_search    0.00  15.92 <0.005    187.21
channel_referral       0.00  -6.39 <0.005     32.51
device_mobile          0.00  46.13 <0.005       inf
device_tablet          0.00   9.39 <0.005     67.11
---
Concordance = 0.60
Partial AIC = 690433.62
log-likelihood ratio test = 3527.20 on 6 df
-log2(p) of ll-ratio test = inf

In [7]:
fig, ax = plt.subplots(figsize=(8, 5))
cph.plot(ax=ax)
ax.set_title("Cox PH: Hazard Ratios (>1 = higher abandonment risk)", fontsize=12)
ax.axvline(0, color="gray", linestyle="--", linewidth=0.8)
plt.tight_layout()
plt.savefig(ASSETS_DIR / "cox_hazard_ratios.png", dpi=150)
plt.close()

## Key Takeaways

- Survival drops sharpest between **driver → quotes** and **quotes → bind**
- `paid_search` has the **highest hazard ratio** among channels (users are window shoppers)
- `mobile` device increases abandonment risk by ~30% vs desktop
- `prior_insured` users survive significantly longer — they understand the product